In [0]:
%sql
USE CATALOG workspace;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS project_21b;

In [0]:
%sql
USE SCHEMA project_21b;

In [0]:
%sql
SELECT CURRENT_CATALOG(),CURRENT_SCHEMA();

In [0]:
%sql
CREATE TABLE IF NOT EXISTS raw_clicks_stg (
    click_id VARCHAR(50),
    user_id VARCHAR(50),
    sku VARCHAR(50),
    action VARCHAR(50)
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS raw_orders_stg (
    order_id VARCHAR(50) PRIMARY KEY,
    user_id VARCHAR(50),
    sku VARCHAR(50),
    price DECIMAL(10, 2)
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS raw_returns_stg (
    return_id VARCHAR(50) PRIMARY KEY,
    order_id VARCHAR(50),
    reason VARCHAR(100)
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS DIM_CUSTOMER_CONFORMED (
    user_sk VARCHAR(50) PRIMARY KEY,
    natural_usr_id VARCHAR(50),
    conformed_status VARCHAR(20)
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS FACT_ORDERS_CDC (
    order_id VARCHAR(50) PRIMARY KEY,
    user_id VARCHAR(50),
    price DECIMAL(10, 2)
);

In [0]:
%sql
INSERT INTO DIM_CUSTOMER_CONFORMED VALUES ('USR_SK_9001', 'USR-1', 'VERIFIED');


In [0]:
%sql
INSERT INTO raw_clicks_stg VALUES ('CLK-90', 'USR-1', 'SKU-A', 'view');


In [0]:
%sql
INSERT INTO raw_orders_stg VALUES ('ORD-50', 'USR-1', 'SKU-A', 49.99);


In [0]:
%sql
INSERT INTO raw_returns_stg VALUES ('RET-10', 'ORD-50', 'damaged');


In [0]:
%sql
SELECT  'CLICKSTREAM' AS SOURCE_STREAM,
        COUNT(*) AS RECORDS_PASSED
FROM raw_clicks_stg
UNION ALL
SELECT  'ORDERS',
        COUNT(*)
FROM raw_orders_stg
UNION ALL
SELECT 'RETURNS',
        COUNT(*)
FROM raw_returns_stg;

In [0]:
%sql
SELECT  'Clickstream' AS PROCESS,
        CASE WHEN COUNT(DISTINCT C.USER_ID)=COUNT(DIM.NATURAL_USR_ID)
        THEN 'YES'
        ELSE 'NO'
        END AS CONFORMED_USR,
        'Event-Level' AS GRAIN_LEVEL,
        'Traffic Mart' AS MART_OWNER
FROM raw_clicks_stg C
LEFT JOIN DIM_CUSTOMER_CONFORMED DIM
ON C.USER_ID = DIM.NATURAL_USR_ID
UNION ALL
SELECT  'Orders' AS PROCESS,
        CASE WHEN COUNT(DISTINCT O.USER_ID)=COUNT(DIM.NATURAL_USR_ID)
        THEN 'YES'
        ELSE 'NO'
        END AS CONFORMED_USR,
        'Line-Item' AS GRAIN_LEVEL,
        'Revenue Mart' AS MART_OWNER
FROM raw_orders_stg O
LEFT JOIN DIM_CUSTOMER_CONFORMED DIM
ON O.USER_ID = DIM.NATURAL_USR_ID;

In [0]:
%sql
SELECT  USER_SK,
        NATURAL_USR_ID,
        conformed_status
FROM DIM_CUSTOMER_CONFORMED
WHERE conformed_status = 'VERIFIED';

In [0]:
%sql
CREATE VIEW VW_ORDERS_CROSS_MART AS
SELECT  ORD.ORDER_ID,
        USER_SK,
        ORD.PRICE,
        CASE WHEN RET.ORDER_ID IS NOT NULL 
        THEN 'YES'
        ELSE 'NO'
        END AS RETURNED,
        RET.REASON
FROM raw_orders_stg ORD
JOIN raw_returns_stg RET
ON ORD.ORDER_ID = RET.ORDER_ID
JOIN dim_customer_conformed DIM
ON ORD.USER_ID=DIM.natural_usr_id;

In [0]:
%sql
SELECT * 
FROM VW_ORDERS_CROSS_MART;

In [0]:
%sql
MERGE INTO FACT_ORDERS_CDC T
USING (SELECT ORDER_ID,USER_ID,PRICE FROM raw_orders_stg) S
ON T.ORDER_ID = S.ORDER_ID
WHEN MATCHED THEN
    UPDATE 
        SET T.USER_ID = S.USER_ID,
            T.PRICE = S.PRICE
WHEN NOT MATCHED THEN
    INSERT (ORDER_ID,USER_ID,PRICE)
    VALUES (S.ORDER_ID,S.USER_ID,S.PRICE);


In [0]:
%sql
SELECT *
FROM FACT_ORDERS_CDC;

In [0]:
%sql
SELECT 'UNMATCHED_RETURNS' AS AUDIT_CHECK,
        COUNT(*) AS PASS_STATUS
FROM raw_returns_stg RET 
LEFT JOIN raw_orders_stg ORD
ON ORD.order_id = RET.order_id
WHERE ORD.order_id IS NULL;

In [0]:
%sql
DESCRIBE HISTORY FACT_ORDERS_CDC;

In [0]:
%sql
SELECT  ORDER_ID,
        PRICE
FROM fact_orders_cdc VERSION AS OF 1;

In [0]:
%sql
CREATE TABLE FACT_CLICKSTREAM 
LIKE RAW_CLICKS_STG;
    
INSERT INTO FACT_CLICKSTREAM
SELECT *
FROM RAW_CLICKS_STG;

In [0]:
%sql
OPTIMIZE FACT_CLICKSTREAM 
ZORDER BY (USER_ID,SKU);

In [0]:
%sql
DESCRIBE HISTORY FACT_CLICKSTREAM;

In [0]:
%sql
SELECT 'FACT_CLICKSTREAM' AS MART_TABLE,
        'SUCCESS' AS OPTIMIZATION_RESULT;